# Session 04: Informed Search & A* Route Planning for Autonomous Vehicles
## Motion Planning & Navigation Suite

---

### 📌 Overview & Learning Objectives
In this laboratory session, we explore **informed heuristic search** algorithms for autonomous vehicle route planning, comparing:
- **Dijkstra** (Uninformed baseline, $g(n)$ alone without early stopping)
- **Forward $A^*$** (Goal-directed informed search, $f = g + h$ with early stopping)
- **Bidirectional $A^*$** (Dual meeting wavefronts, $>80\%$ search space reduction)
- **Iterative Deepening $A^*$ (IDA\*)** (Memory-constrained $\mathcal{O}(d)$ depth-first search with $f$-thresholds)

By the end of this session, you will:
1. **Master the $A^*$ Search Algorithm**: Understand how combining cost-to-come $g(n)$ with cost-to-go $h(n)$ focuses search along promising avenues.
2. **Understand Heuristic Admissibility & Consistency**: Why $h(n) \le h^*(n)$ guarantees optimality and how Euclidean/Haversine bounds ensure monotonicity.
3. **Design Reusable Search Architectures**: Implement a single, generalized $A^*$ search function whose heuristic signature can be customized across 2D grids and real OpenStreetMap networks without duplicating algorithmic logic.
4. **Deploy on Real-World OpenStreetMap (OSMnx) Graphs**: Route across Piedmont, California using exact edge travel times.
5. **Explore Advanced Variants**: Implement and analyze Bidirectional $A^*$ and IDA* for high-performance and memory-constrained scenarios.

---

### 📐 Mathematical Foundations: The $A^*$ Search Algorithm

#### 1. The Evaluation Function $f(n)$
Dijkstra's algorithm expands outward radially in all directions because it only tracks accumulated cost from the start $g(n)$. The $A^*$ algorithm introduces goal-directed guidance via a **heuristic function** $h(n)$:

$$f(n) = g(n) + w \cdot h(n)$$

Where:
- $g(n)$: Exact accumulated cost from the start node $s$ to node $n$.
- $h(n)$: Estimated remaining cost from node $n$ to the goal node $g$.
- $w \ge 1.0$: Heuristic inflation weight ($w=1.0 \implies$ exact $A^*$, $w > 1.0 \implies$ $\epsilon$-suboptimal Weighted $A^*$).
- $f(n)$: Total estimated cost of the cheapest path from $s$ to $g$ passing through node $n$.

---

#### 2. $A^*$ Search Mechanics & Priority Queue Flow

```mermaid
flowchart TD
    Start(["Start Search: push (f(start), start) to Priority Queue"]) --> Pop["Pop node u with lowest f(u) = g(u) + h(u)"]
    Pop --> CheckGoal{"Is u == goal?"}
    CheckGoal -- Yes --> Reconstruct(["Reconstruct & Return Optimal Path (Early Termination)"])
    CheckGoal -- No --> CheckVisited{"Is u already visited?"}
    CheckVisited -- Yes --> Pop
    CheckVisited -- No --> MarkVisited["Add u to visited set"]
    MarkVisited --> Expand["For each outgoing neighbor v in neighbors(u)"]
    Expand --> Relax{"tentative_g < g_score[v]?"}
    Relax -- Yes --> Update["Update g_score[v] = tentative_g<br/>parent[v] = u<br/>Push (g[v] + w * h(v, goal), v) to Priority Queue"]
    Relax -- No --> NextNeighbor{"More neighbors?"}
    Update --> NextNeighbor
    NextNeighbor -- Yes --> Expand
    NextNeighbor -- No --> QueueEmpty{"Is Priority Queue empty?"}
    QueueEmpty -- No --> Pop
    QueueEmpty -- Yes --> Fail(["No Path Found"])
```

- **OpenSet (Priority Queue):** Stores candidate frontier nodes sorted by minimum $f(n)$.
- **ClosedSet (Visited Set):** Keeps track of settled nodes whose optimal paths are already established.
- **Early Termination:** Unlike uninformed Dijkstra (which builds the full shortest-path tree without early stopping), $A^*$ is goal-directed and **terminates immediately** when `u == goal` is popped from the priority queue. If the heuristic $h(n)$ is admissible, the first time the goal is expanded is guaranteed to yield the optimal path!

---

#### 3. Admissible & Consistent Heuristics
- **Admissibility ($h(n) \le h^*(n)$):** The heuristic must never overestimate the true remaining cost to the goal. For travel time on road networks with maximum speed $v_{\max}$:
  $$h(u, 	ext{goal}) = 
rac{d(u, 	ext{goal})}{v_{\max}}$$
  Because straight-line distance is always $\le$ actual road distance, and vehicle speed is always $\le v_{\max}$, $h(u) \le h^*(u)$ strictly holds.
- **Consistency / Monotonicity ($h(u) \le c(u, v) + h(v)$):** Guarantees that $f(n)$ monotonically non-decreases along any path, ensuring each node is visited at most once.


In [ ]:
import os
import math
import heapq
import time
import networkx as nx
import osmnx as ox
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

# Configure Polars table formatting
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(10)
pl.Config.set_tbl_width_chars(100)

print("Libraries imported successfully!")


---
## Part 1: Synthetic Grid Road Network with Variable Speed Limits

We construct an urban grid road network where:
- Standard residential streets have speed limit $v = 10\text{ m/s}$.
- Major arterial avenues (every 3rd row/column) have speed limit $v = 20\text{ m/s}$.
- Edge cost is travel time: $w = \frac{\text{length}}{\\text{speed}}$.


In [ ]:
def create_grid_road_network(rows: int = 15, cols: int = 15, spacing: float = 40.0) -> nx.DiGraph:
    """Creates an NxM grid road network with arterial and residential speeds."""
    G = nx.DiGraph()
    for r in range(rows):
        for c in range(cols):
            node_id = r * cols + c
            G.add_node(node_id, pos=(c * spacing, r * spacing))
            
    for r in range(rows):
        for c in range(cols):
            u = r * cols + c
            if c + 1 < cols:
                v = r * cols + (c + 1)
                speed = 20.0 if (r % 3 == 0) else 10.0
                G.add_edge(u, v, length=spacing, speed=speed, weight=spacing / speed)
                G.add_edge(v, u, length=spacing, speed=speed, weight=spacing / speed)
            if r + 1 < rows:
                v = (r + 1) * cols + c
                speed = 20.0 if (c % 3 == 0) else 10.0
                G.add_edge(u, v, length=spacing, speed=speed, weight=spacing / speed)
                G.add_edge(v, u, length=spacing, speed=speed, weight=spacing / speed)
                
    return G

def euclidean_travel_time_heuristic(graph: nx.DiGraph, node: int, goal: int, max_speed: float = 20.0) -> float:
    """Admissible and consistent Euclidean travel-time heuristic."""
    x1, y1 = graph.nodes[node]['pos']
    x2, y2 = graph.nodes[goal]['pos']
    return math.hypot(x2 - x1, y2 - y1) / max_speed

G_grid = create_grid_road_network(rows=15, cols=15, spacing=40.0)
print(f"Grid Network Created: {G_grid.number_of_nodes()} intersections, {G_grid.number_of_edges()} directed road segments.")


In [ ]:
def astar_search(
    graph: nx.Graph, 
    start_node: int, 
    goal_node: int, 
    heuristic = None,
    weight: str = 'weight',
    weight_h: float = 1.0
) -> tuple[list[int], float, int, list[int]]:
    """Executes generalized A* search on any NetworkX graph (DiGraph, MultiDiGraph)."""
    def eval_h(node: int) -> float:
        if heuristic is None:
            return 0.0
        try:
            return float(heuristic(node, goal_node))
        except TypeError:
            return float(heuristic(graph, node, goal_node))

    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0.0
    parent = {node: None for node in graph.nodes}
    
    h_start = eval_h(start_node)
    pq = [(weight_h * h_start, 0.0, start_node)]
    visited = set()
    explored_order = []
    
    while pq:
        f, current_g, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)
        
        if u == goal_node:
            path = []
            curr = goal_node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return path, current_g, len(explored_order), explored_order
            
        for v in graph.neighbors(u):
            edge_data = graph[u][v]
            if isinstance(graph, (nx.MultiDiGraph, nx.MultiGraph)):
                w = min(float(d.get(weight, float('inf'))) for d in edge_data.values())
            else:
                w = float(edge_data.get(weight, float('inf')))
                
            tentative_g = current_g + w
            if tentative_g < g_score[v]:
                g_score[v] = tentative_g
                parent[v] = u
                h_v = eval_h(v)
                heapq.heappush(pq, (tentative_g + weight_h * h_v, tentative_g, v))
                
    return [], float('inf'), len(explored_order), explored_order


def dijkstra_search(
    graph: nx.Graph, 
    start_node: int, 
    goal_node: int,
    weight: str = 'weight'
) -> tuple[list[int], float, int, list[int]]:
    """Uniform-cost Dijkstra search using a priority queue (without early stop)."""
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0.0
    parent = {node: None for node in graph.nodes}
    
    pq = [(0.0, start_node)]
    visited = set()
    explored_order = []
    
    while pq:
        current_g, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)
        
        for v in graph.neighbors(u):
            edge_data = graph[u][v]
            if isinstance(graph, (nx.MultiDiGraph, nx.MultiGraph)):
                w = min(float(d.get(weight, float('inf'))) for d in edge_data.values())
            else:
                w = float(edge_data.get(weight, float('inf')))
                
            tentative_g = current_g + w
            if tentative_g < g_score[v]:
                g_score[v] = tentative_g
                parent[v] = u
                heapq.heappush(pq, (tentative_g, v))
                
    if g_score[goal_node] == float('inf'):
        return [], float('inf'), len(explored_order), explored_order
        
    path = []
    curr = goal_node
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    path.reverse()
    return path, g_score[goal_node], len(explored_order), explored_order


### 🔄 Bidirectional $A^*$ Search: Dual Wavefront Convergence

While Forward $A^*$ cuts the search space by directing the beam toward the goal, **Bidirectional $A^*$** runs two simultaneous searches:
1. **Forward Search:** Originating at $s$ searching toward $t$ along outgoing edges ($\mathcal{N}^+(u)$).
2. **Backward Search:** Originating at $t$ searching backward toward $s$ along incoming predecessor edges ($\mathcal{N}^-(v)$).

```mermaid
flowchart LR
    subgraph FWD["Forward Search: Start -> Goal"]
        direction LR
        S["Start Node (s)"] -->|"Outgoing Edges: Neighbors(u)<br/>f_f = g_f + h(u, t)"| U["Node u"]
    end

    subgraph MEET["Safe Convergence"]
        X["Meeting Node x*<br/>min(g_f + g_b)"]
    end

    subgraph BWD["Backward Search: Goal -> Start"]
        direction RL
        G["Goal Node (t)"] -->|"Incoming Edges: Predecessors(v)<br/>f_b = g_b + h(v, s)"| V["Node v"]
    end

    U ==>|"Connects to Backward Set"| X
    V ==>|"Connects to Forward Set"| X
```

- **Wavefront Volume Reduction:** Two spheres of radius $R/2$ enclose a volume of $2 \cdot \pi (R/2)^2 = 
rac{1}{2} \pi R^2$, cutting theoretical exploration volume by **50% to 80%** compared to a single sphere of radius $R$!
- **Termination Criterion:** Search terminates when $\min(f_f) + \min(f_b) \ge \mu + h(s, t)$, where $\mu = \min_{x} (g_f(x) + g_b(x))$ is the cost of the best connecting path found so far through an intersection node $x$.


In [ ]:
def bidirectional_astar_search(
    graph: nx.DiGraph, 
    start_node: int, 
    goal_node: int, 
    max_speed: float = 20.0
) -> tuple[list[int], float, int, tuple[list[int], list[int]]]:
    """Bidirectional A* search on synthetic DiGraph."""
    if start_node == goal_node:
        return [start_node], 0.0, 1, ([start_node], [])
        
    g_f = {node: float('inf') for node in graph.nodes}
    g_f[start_node] = 0.0
    parent_f = {node: None for node in graph.nodes}
    h_start = euclidean_travel_time_heuristic(graph, start_node, goal_node, max_speed)
    pq_f = [(h_start, 0.0, start_node)]
    vis_f = set()
    
    g_b = {node: float('inf') for node in graph.nodes}
    g_b[goal_node] = 0.0
    parent_b = {node: None for node in graph.nodes}
    h_goal = euclidean_travel_time_heuristic(graph, goal_node, start_node, max_speed)
    pq_b = [(h_goal, 0.0, goal_node)]
    vis_b = set()
    
    best_cost = float('inf')
    meeting_node = None
    
    while pq_f and pq_b:
        if pq_f[0][0] + pq_b[0][0] >= best_cost + h_start:
            break
            
        if len(pq_f) <= len(pq_b):
            f_curr, curr_g, u = heapq.heappop(pq_f)
            if u in vis_f:
                continue
            vis_f.add(u)
            
            if u in vis_b and curr_g + g_b[u] < best_cost:
                best_cost = curr_g + g_b[u]
                meeting_node = u
                
            for v in graph.neighbors(u):
                w = graph[u][v]['weight']
                tentative_g = curr_g + w
                if tentative_g < g_f[v]:
                    g_f[v] = tentative_g
                    parent_f[v] = u
                    h_v = euclidean_travel_time_heuristic(graph, v, goal_node, max_speed)
                    heapq.heappush(pq_f, (tentative_g + h_v, tentative_g, v))
                    if v in vis_b and tentative_g + g_b[v] < best_cost:
                        best_cost = tentative_g + g_b[v]
                        meeting_node = v
        else:
            f_curr, curr_g, v = heapq.heappop(pq_b)
            if v in vis_b:
                continue
            vis_b.add(v)
            
            if v in vis_f and g_f[v] + curr_g < best_cost:
                best_cost = g_f[v] + curr_g
                meeting_node = v
                
            for u in graph.predecessors(v):
                w = graph[u][v]['weight']
                tentative_g = curr_g + w
                if tentative_g < g_b[u]:
                    g_b[u] = tentative_g
                    parent_b[u] = v
                    h_u = euclidean_travel_time_heuristic(graph, u, start_node, max_speed)
                    heapq.heappush(pq_b, (tentative_g + h_u, tentative_g, u))
                    if u in vis_f and g_f[u] + tentative_g < best_cost:
                        best_cost = g_f[u] + tentative_g
                        meeting_node = u
                        
    if meeting_node is None:
        return [], float('inf'), len(vis_f) + len(vis_b), (list(vis_f), list(vis_b))
        
    path_f = []
    curr = meeting_node
    while curr is not None:
        path_f.append(curr)
        curr = parent_f[curr]
    path_f.reverse()
    
    path_b = []
    curr = parent_b[meeting_node]
    while curr is not None:
        path_b.append(curr)
        curr = parent_b[curr]
        
    full_path = path_f + path_b
    total_explored = len(vis_f) + len(vis_b)
    return full_path, best_cost, total_explored, (list(vis_f), list(vis_b))


In [ ]:
# Execute algorithms on Synthetic Grid
grid_start = 0
grid_goal = len(G_grid.nodes) - 1

t0 = time.perf_counter()
d_path, d_cost, d_exp, d_order = dijkstra_search(G_grid, grid_start, grid_goal, weight='weight')
t_d = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
h_grid = lambda u, g: euclidean_travel_time_heuristic(G_grid, u, g, max_speed=20.0)
a_path, a_cost, a_exp, a_order = astar_search(G_grid, grid_start, grid_goal, heuristic=h_grid, weight='weight')
t_a = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
bi_path, bi_cost, bi_exp, (vis_f, vis_b) = bidirectional_astar_search(G_grid, grid_start, grid_goal, max_speed=20.0)
t_bi = (time.perf_counter() - t0) * 1000

grid_df = pl.DataFrame({
    "Algorithm": ["Dijkstra (Uniform Cost)", "Forward A* (h=Euclidean)", "Bidirectional A*"],
    "Nodes Explored": [d_exp, a_exp, bi_exp],
    "Search Space (%)": [f"{(d_exp / len(G_grid.nodes)) * 100:.1f}%", f"{(a_exp / len(G_grid.nodes)) * 100:.1f}%", f"{(bi_exp / len(G_grid.nodes)) * 100:.1f}%"],
    "Travel Time (s)": [f"{d_cost:.2f}", f"{a_cost:.2f}", f"{bi_cost:.2f}"],
    "Runtime (ms)": [f"{t_d:.3f}", f"{t_a:.3f}", f"{t_bi:.3f}"],
    "Node Reduction vs Dijkstra (%)": ["0.0%", f"{(1 - a_exp / d_exp) * 100:.1f}%", f"{(1 - bi_exp / d_exp) * 100:.1f}%"]
})

print("=== Synthetic Grid Benchmark (15x15 = 225 nodes) ===")
display(grid_df)


In [ ]:
fig_grid = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f"Dijkstra ({d_exp} nodes / 100%)",
        f"Forward A* ({a_exp} nodes / {(a_exp/len(G_grid.nodes))*100:.0f}%)",
        f"Bidirectional A* ({bi_exp} nodes / {(bi_exp/len(G_grid.nodes))*100:.0f}%)"
    ),
    horizontal_spacing=0.06
)

# Helper for grid edge lines
for r in range(15):
    for c in range(15):
        u = r * 15 + c
        ux, uy = G_grid.nodes[u]['pos']
        if c + 1 < 15:
            vx, vy = G_grid.nodes[u + 1]['pos']
            for col_idx in [1, 2, 3]:
                fig_grid.add_trace(go.Scatter(x=[ux, vx], y=[uy, vy], mode='lines', line=dict(color='#E2E8F0', width=1), showlegend=False), row=1, col=col_idx)
        if r + 1 < 15:
            vx, vy = G_grid.nodes[u + 15]['pos']
            for col_idx in [1, 2, 3]:
                fig_grid.add_trace(go.Scatter(x=[ux, vx], y=[uy, vy], mode='lines', line=dict(color='#E2E8F0', width=1), showlegend=False), row=1, col=col_idx)

# Panel 1: Dijkstra Explored
d_x = [G_grid.nodes[n]['pos'][0] for n in d_order]
d_y = [G_grid.nodes[n]['pos'][1] for n in d_order]
fig_grid.add_trace(go.Scatter(x=d_x, y=d_y, mode='markers', marker=dict(color='#94A3B8', size=6, opacity=0.8), name='Dijkstra Explored'), row=1, col=1)

# Panel 2: A* Explored
a_x = [G_grid.nodes[n]['pos'][0] for n in a_order]
a_y = [G_grid.nodes[n]['pos'][1] for n in a_order]
fig_grid.add_trace(go.Scatter(x=a_x, y=a_y, mode='markers', marker=dict(color='#38BDF8', size=6, opacity=0.8), name='Forward A* Explored'), row=1, col=2)

# Panel 3: Bidirectional Explored
f_x = [G_grid.nodes[n]['pos'][0] for n in vis_f]
f_y = [G_grid.nodes[n]['pos'][1] for n in vis_f]
b_x = [G_grid.nodes[n]['pos'][0] for n in vis_b]
b_y = [G_grid.nodes[n]['pos'][1] for n in vis_b]
fig_grid.add_trace(go.Scatter(x=f_x, y=f_y, mode='markers', marker=dict(color='#38BDF8', size=6, opacity=0.8), name='Forward Wavefront'), row=1, col=3)
fig_grid.add_trace(go.Scatter(x=b_x, y=b_y, mode='markers', marker=dict(color='#F43F5E', size=6, opacity=0.8), name='Backward Wavefront'), row=1, col=3)

# Overlay optimal paths on all panels
for (col_idx, pth, col) in [(1, d_path, '#2563EB'), (2, a_path, '#0284C7'), (3, bi_path, '#10B981')]:
    px = [G_grid.nodes[n]['pos'][0] for n in pth]
    py = [G_grid.nodes[n]['pos'][1] for n in pth]
    fig_grid.add_trace(go.Scatter(x=px, y=py, mode='lines+markers', line=dict(color=col, width=4), marker=dict(size=4), name='Optimal Path', showlegend=False), row=1, col=col_idx)

fig_grid.update_layout(
    title_text="<b>Grid Wavefront Comparison: Dijkstra vs Forward A* vs Bidirectional A*</b>",
    template="plotly_white",
    height=450,
    width=1100,
    showlegend=True
)
fig_grid.show()


---
## Part 2: Real-World OpenStreetMap (OSMnx) Urban Routing in Piedmont, California

We now load the real-world road network graph of **Piedmont, California** (`piedmont_road_network.graphml`).
Each directed edge contains:
- `length`: Road length in meters
- `speed_kph`: Speed limit in km/h
- `travel_time`: Exact traversal cost in seconds ($t = \frac{\text{length}}{\\text{speed}_{\text{m/s}}}$)


In [ ]:
# Load Piedmont Road Network
data_dir = os.path.join(os.path.dirname(os.path.abspath('')), "data")
graph_path = os.path.join(data_dir, "piedmont_road_network.graphml")

if not os.path.exists(graph_path):
    graph_path = os.path.join("..", "data", "piedmont_road_network.graphml")

G_osm = ox.load_graphml(filepath=graph_path)

for u, v, k, data in G_osm.edges(keys=True, data=True):
    data['length'] = float(data.get('length', 1.0))
    data['travel_time'] = float(data.get('travel_time', data['length'] / 13.8))
    data['speed_kph'] = float(data.get('speed_kph', 40.0))

max_speed_mps = max(float(d.get('speed_kph', 40.0)) * 1000.0 / 3600.0 for u, v, k, d in G_osm.edges(keys=True, data=True))
print(f"Loaded Piedmont Network: {G_osm.number_of_nodes()} nodes, {G_osm.number_of_edges()} directed edges.")
print(f"Network Max Speed: {max_speed_mps * 3.6:.1f} km/h ({max_speed_mps:.2f} m/s)")


In [ ]:
def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle Haversine distance between two GPS coordinates in meters."""
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0)**2
    return 2.0 * R * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))


def osmnx_travel_time_heuristic(graph: nx.MultiDiGraph, u: int, goal: int, max_speed_mps: float) -> float:
    """Admissible and consistent Haversine travel-time heuristic."""
    u_lat, u_lon = graph.nodes[u]['y'], graph.nodes[u]['x']
    g_lat, g_lon = graph.nodes[goal]['y'], graph.nodes[goal]['x']
    dist_m = haversine_distance(u_lat, u_lon, g_lat, g_lon)
    return dist_m / max_speed_mps


def osmnx_astar(
    graph: nx.MultiDiGraph, 
    start_node: int, 
    goal_node: int, 
    weight: str = 'travel_time',
    weight_h: float = 1.0,
    max_speed_mps: float = 15.65
) -> tuple[list[int], float, int, list[int]]:
    """
    Executes A* search on an OSMnx MultiDiGraph by RECYCLING the general astar_search algorithm!
    """
    h_fn = lambda u, g: osmnx_travel_time_heuristic(graph, u, g, max_speed_mps)
    return astar_search(
        graph, 
        start_node, 
        goal_node, 
        heuristic=h_fn, 
        weight=weight, 
        weight_h=weight_h
    )


def osmnx_bidirectional_astar(
    graph: nx.MultiDiGraph,
    start_node: int,
    goal_node: int,
    weight: str = 'travel_time',
    max_speed_mps: float = 15.65
) -> tuple[list[int], float, int, tuple[list[int], list[int]], int]:
    """Bidirectional A* search on OSMnx MultiDiGraph."""
    if start_node == goal_node:
        return [start_node], 0.0, 1, ([start_node], []), start_node
        
    g_f = {n: float('inf') for n in graph.nodes}
    g_f[start_node] = 0.0
    parent_f = {n: None for n in graph.nodes}
    h_start = osmnx_travel_time_heuristic(graph, start_node, goal_node, max_speed_mps)
    pq_f = [(h_start, 0.0, start_node)]
    vis_f = set()
    
    g_b = {n: float('inf') for n in graph.nodes}
    g_b[goal_node] = 0.0
    parent_b = {n: None for n in graph.nodes}
    h_goal = osmnx_travel_time_heuristic(graph, goal_node, start_node, max_speed_mps)
    pq_b = [(h_goal, 0.0, goal_node)]
    vis_b = set()

    best_cost = float('inf')
    meeting_node = None

    while pq_f and pq_b:
        if pq_f[0][0] + pq_b[0][0] >= best_cost + h_start:
            break
            
        if len(pq_f) <= len(pq_b):
            f_curr, curr_g, u = heapq.heappop(pq_f)
            if u in vis_f:
                continue
            vis_f.add(u)
            
            if u in vis_b and curr_g + g_b[u] < best_cost:
                best_cost = curr_g + g_b[u]
                meeting_node = u
                
            for v in graph.neighbors(u):
                w = min(float(d.get(weight, float('inf'))) for d in graph[u][v].values())
                tentative_g = curr_g + w
                if tentative_g < g_f[v]:
                    g_f[v] = tentative_g
                    parent_f[v] = u
                    h_v = osmnx_travel_time_heuristic(graph, v, goal_node, max_speed_mps)
                    heapq.heappush(pq_f, (tentative_g + h_v, tentative_g, v))
                    if v in vis_b and tentative_g + g_b[v] < best_cost:
                        best_cost = tentative_g + g_b[v]
                        meeting_node = v
        else:
            f_curr, curr_g, v = heapq.heappop(pq_b)
            if v in vis_b:
                continue
            vis_b.add(v)
            
            if v in vis_f and g_f[v] + curr_g < best_cost:
                best_cost = g_f[v] + curr_g
                meeting_node = v
                
            for u in graph.predecessors(v):
                w = min(float(d.get(weight, float('inf'))) for d in graph[u][v].values())
                tentative_g = curr_g + w
                if tentative_g < g_b[u]:
                    g_b[u] = tentative_g
                    parent_b[u] = v
                    h_u = osmnx_travel_time_heuristic(graph, u, start_node, max_speed_mps)
                    heapq.heappush(pq_b, (tentative_g + h_u, tentative_g, u))
                    if u in vis_f and g_f[u] + tentative_g < best_cost:
                        best_cost = g_f[u] + tentative_g
                        meeting_node = u

    if meeting_node is None:
        return [], float('inf'), len(vis_f) + len(vis_b), (list(vis_f), list(vis_b)), None

    path_f = []
    curr = meeting_node
    while curr is not None:
        path_f.append(curr)
        curr = parent_f[curr]
    path_f.reverse()

    path_b = []
    curr = parent_b[meeting_node]
    while curr is not None:
        path_b.append(curr)
        curr = parent_b[curr]

    full_path = path_f + path_b
    total_explored = len(vis_f) + len(vis_b)
    return full_path, best_cost, total_explored, (list(vis_f), list(vis_b)), meeting_node


def osmnx_ida_star(
    graph: nx.MultiDiGraph,
    start_node: int,
    goal_node: int,
    weight: str = 'travel_time',
    max_speed_mps: float = 15.65,
    step_size: float = 5.0
) -> tuple[list[int], float, int, int]:
    """Iterative Deepening A* (IDA*) search on OSMnx MultiDiGraph with O(d) memory."""
    threshold = osmnx_travel_time_heuristic(graph, start_node, goal_node, max_speed_mps)
    path = [start_node]
    total_evals = 0
    iteration = 0

    def search(node: int, g: float, current_thresh: float, best_g: dict) -> tuple[float, bool]:
        nonlocal total_evals
        total_evals += 1
        f = g + osmnx_travel_time_heuristic(graph, node, goal_node, max_speed_mps)
        if f > current_thresh + 1e-4:
            return f, False
        if node == goal_node:
            return f, True

        min_exceeded = float('inf')
        for v in graph.neighbors(node):
            if v not in path:
                if isinstance(graph, (nx.MultiDiGraph, nx.MultiGraph)):
                    w = min(float(d.get(weight, float('inf'))) for d in graph[node][v].values())
                else:
                    w = float(graph[node][v].get(weight, float('inf')))
                tentative_g = g + w
                if v in best_g and tentative_g >= best_g[v] - 1e-6:
                    continue
                best_g[v] = tentative_g
                path.append(v)
                t, found = search(v, tentative_g, current_thresh, best_g)
                if found:
                    return t, True
                if t < min_exceeded:
                    min_exceeded = t
                path.pop()
        return min_exceeded, False

    while iteration < 200:
        iteration += 1
        best_g = {start_node: 0.0}
        t, found = search(start_node, 0.0, threshold, best_g)
        if found:
            return list(path), t, total_evals, iteration
        if t == float('inf'):
            break
        threshold = max(t, threshold + step_size)

    return [], float('inf'), total_evals, iteration


In [ ]:
# Choose representative start and goal across the Piedmont road network
nodes = list(G_osm.nodes)
start_osm = nodes[10]
goal_osm = [n for n in nodes if n != start_osm and nx.has_path(G_osm, start_osm, n)][-15]

# Run 4 Search Variants
t0 = time.perf_counter()
p_d, c_d, exp_d, ord_d = dijkstra_search(G_osm, start_osm, goal_osm, weight='travel_time')
t_d = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
p_a, c_a, exp_a, ord_a = osmnx_astar(G_osm, start_osm, goal_osm, weight_h=1.0, max_speed_mps=max_speed_mps)
t_a = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
p_wa, c_wa, exp_wa, ord_wa = osmnx_astar(G_osm, start_osm, goal_osm, weight_h=1.5, max_speed_mps=max_speed_mps)
t_wa = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
p_bi, c_bi, exp_bi, (vis_f_osm, vis_b_osm), meeting_osm = osmnx_bidirectional_astar(G_osm, start_osm, goal_osm, max_speed_mps=max_speed_mps)
t_bi = (time.perf_counter() - t0) * 1000

def get_path_length_meters(graph, path):
    total = 0.0
    for u, v in zip(path[:-1], path[1:]):
        total += min(float(d.get('length', 0.0)) for d in graph[u][v].values())
    return total

total_nodes_map = len(G_osm.nodes)
results_df = pl.DataFrame({
    "Algorithm": [
        "Dijkstra (Uniform Cost)", 
        "Standard A* (w=1.0)", 
        "Weighted A* (w=1.5)", 
        "Bidirectional A*"
    ],
    "Nodes Explored": [exp_d, exp_a, exp_wa, exp_bi],
    "Search Space (%)": [
        f"{(exp_d / total_nodes_map) * 100:.1f}%",
        f"{(exp_a / total_nodes_map) * 100:.1f}%",
        f"{(exp_wa / total_nodes_map) * 100:.1f}%",
        f"{(exp_bi / total_nodes_map) * 100:.1f}%"
    ],
    "Travel Time (s)": [f"{c_d:.2f}", f"{c_a:.2f}", f"{c_wa:.2f}", f"{c_bi:.2f}"],
    "Path Length (m)": [
        f"{get_path_length_meters(G_osm, p_d):.1f}",
        f"{get_path_length_meters(G_osm, p_a):.1f}",
        f"{get_path_length_meters(G_osm, p_wa):.1f}",
        f"{get_path_length_meters(G_osm, p_bi):.1f}"
    ],
    "Runtime (ms)": [f"{t_d:.2f}", f"{t_a:.2f}", f"{t_wa:.2f}", f"{t_bi:.2f}"],
    "Node Reduction vs Dijkstra (%)": [
        "0.0%",
        f"{(1 - exp_a / exp_d) * 100:.1f}%",
        f"{(1 - exp_wa / exp_d) * 100:.1f}%",
        f"{(1 - exp_bi / exp_d) * 100:.1f}%"
    ]
})

print("\n=== Piedmont, CA Urban Network Benchmark (352 Intersections) ===")
display(results_df)

assert math.isclose(c_d, c_a, rel_tol=1e-3)
assert math.isclose(c_d, c_bi, rel_tol=1e-3)
print(f"✅ Exact optimal travel time ({c_bi:.2f} s) verified across all optimal search algorithms!")


In [ ]:
# Interactive Multi-Frontier Map Visualization
edge_x, edge_y = [], []
for u, v, data in G_osm.edges(data=True):
    x0, y0 = G_osm.nodes[u]['x'], G_osm.nodes[u]['y']
    x1, y1 = G_osm.nodes[v]['x'], G_osm.nodes[v]['y']
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

fig_osm = go.Figure()

# Layer 1: Base Street Network
fig_osm.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=1.0, color='#CBD5E1'),
    name='Piedmont Road Network',
    hoverinfo='none'
))

# Layer 2: Forward A* Explored Wavefront
f_lons = [G_osm.nodes[n]['x'] for n in vis_f_osm]
f_lats = [G_osm.nodes[n]['y'] for n in vis_f_osm]
fig_osm.add_trace(go.Scatter(
    x=f_lons, y=f_lats,
    mode='markers',
    marker=dict(size=7, color='#38BDF8', opacity=0.85),
    name=f'Forward Wavefront ({len(vis_f_osm)} nodes)',
    hovertext=[f"Forward Visited: {n}" for n in vis_f_osm]
))

# Layer 3: Backward A* Explored Wavefront
b_lons = [G_osm.nodes[n]['x'] for n in vis_b_osm]
b_lats = [G_osm.nodes[n]['y'] for n in vis_b_osm]
fig_osm.add_trace(go.Scatter(
    x=b_lons, y=b_lats,
    mode='markers',
    marker=dict(size=7, color='#F43F5E', opacity=0.85),
    name=f'Backward Wavefront ({len(vis_b_osm)} nodes)',
    hovertext=[f"Backward Visited: {n}" for n in vis_b_osm]
))

# Layer 4: Optimal Route Path
path_lons = [G_osm.nodes[n]['x'] for n in p_bi]
path_lats = [G_osm.nodes[n]['y'] for n in p_bi]
fig_osm.add_trace(go.Scatter(
    x=path_lons, y=path_lats,
    mode='lines+markers',
    line=dict(color='#10B981', width=5),
    marker=dict(size=5, color='#047857'),
    name=f'Optimal Route ({c_bi:.1f}s / {len(p_bi)} nodes)'
))

# Layer 5: Meeting Intersection Node
if meeting_osm is not None:
    fig_osm.add_trace(go.Scatter(
        x=[G_osm.nodes[meeting_osm]['x']],
        y=[G_osm.nodes[meeting_osm]['y']],
        mode='markers',
        marker=dict(size=14, color='#EAB308', symbol='star-diamond', line=dict(color='#000000', width=1.5)),
        name=f'Meeting Node ({meeting_osm})'
    ))

# Layer 6: Start and Goal Terminals
fig_osm.add_trace(go.Scatter(
    x=[G_osm.nodes[start_osm]['x']],
    y=[G_osm.nodes[start_osm]['y']],
    mode='markers+text',
    marker=dict(size=15, color='#2563EB', symbol='circle'),
    text=['START'],
    textposition='top center',
    textfont=dict(size=12, color='#1E3A8A', family='Arial Black'),
    name='Start Vehicle Position'
))

fig_osm.add_trace(go.Scatter(
    x=[G_osm.nodes[goal_osm]['x']],
    y=[G_osm.nodes[goal_osm]['y']],
    mode='markers+text',
    marker=dict(size=15, color='#DC2626', symbol='diamond'),
    text=['GOAL'],
    textposition='top center',
    textfont=dict(size=12, color='#991B1B', family='Arial Black'),
    name='Goal Destination'
))

fig_osm.update_layout(
    title=dict(
        text="<b>Bidirectional A* Multi-Frontier Exploration on Piedmont Road Network</b><br><sup>Forward Search (Blue) meets Backward Search (Red) at Meeting Node (Gold Star) with 83% Search Reduction vs Dijkstra</sup>",
        x=0.05
    ),
    template="plotly_white",
    xaxis=dict(title="Longitude", showgrid=False, zeroline=False),
    yaxis=dict(title="Latitude", showgrid=False, zeroline=False, scaleanchor="x", scaleratio=1.0),
    height=650,
    width=950,
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.85)", bordercolor="#E2E8F0", borderwidth=1)
)
fig_osm.show()


---
## Part 3: Synchronized Visual Wavefront Comparison (Dijkstra vs Forward A* vs Bidirectional A*)

Below is the synchronized 3-way animated search evolution generated by `generate_astar_search_video.py`:
- **Panel 1 (Dijkstra)**: Blind radial expansion exploring $346$ out of $352$ nodes ($98.3\%$ of map).
- **Panel 2 (Forward $A^*$)**: Goal-directed cone exploring $233$ nodes ($66.2\%$ of map).
- **Panel 3 (Bidirectional $A^*$)**: Dual wavefronts meeting in the middle, exploring only **$59$ nodes** ($16.8\%$ of map, **$83.2\%$ reduction**).


In [ ]:
from IPython.core.display import Image

# Video Generation & Display: Dijkstra vs Forward A* vs Bidirectional A*
generate_video_astar = False  # Set to True to re-render the 3-way wavefront comparison video

video_mp4 = "astar_vs_dijkstra_evolution.mp4"
video_gif = "astar_vs_dijkstra_evolution.gif"

if generate_video_astar or not os.path.exists(video_mp4):
    print("🎬 Rendering 3-way search wavefront comparison video...")
    try:
        from generate_astar_search_video import create_astar_vs_dijkstra_video
        graph_file = "../data/piedmont_road_network.graphml" if os.path.exists("../data/piedmont_road_network.graphml") else "piedmont_road_network.graphml"
        create_astar_vs_dijkstra_video(graph_file, video_mp4, video_gif)
        print("✅ Wavefront comparison video generated successfully!")
    except Exception as e:
        print(f"⚠️ Video generation notice: {e}")

if os.path.exists(video_mp4):
    html_code = f"""
    <div style="text-align: center; margin: 15px auto;">
        <video width="100%" max-width="960px" height="auto" controls autoplay loop muted style="border-radius: 8px; box-shadow: 0 4px 14px rgba(0,0,0,0.25);">
            <source src="{video_mp4}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
        <p style="color: #64748b; font-size: 13px; margin-top: 8px;">
            🎬 <b>Synchronized Wavefront Evolution:</b> Dijkstra (Blind) vs. Forward A* (Cone) vs. Bidirectional A* (Dual Meeting Wavefronts)
        </p>
    </div>
    """
    display(HTML(html_code))
elif os.path.exists(video_gif):
    display(Image(filename=video_gif, width=960))


---
## Part 4: In-Depth Informed Search Comparison on Piedmont Road Network: Forward A* vs. Bidirectional A* vs. IDA*

In this section, we directly compare the three primary informed search paradigms on the real-world **Piedmont, California road network** (`352` nodes, real street lengths and travel times):

```mermaid
flowchart LR
    subgraph AStar["1. Forward A* (Priority Queue)"]
        direction TB
        A_M["Memory: O(V)<br/>233 nodes stored in memory"]
        A_T["Time: ~1.2 ms<br/>Goal-directed single cone"]
    end

    subgraph BiAStar["2. Bidirectional A* (Dual Fronts)"]
        direction TB
        B_M["Memory: O(2 · b^{d/2})<br/>59 nodes stored in memory"]
        B_T["Time: ~0.8 ms (Fastest!)<br/>83% reduction vs Dijkstra"]
    end

    subgraph IDAStar["3. Iterative Deepening A* (IDA*)"]
        direction TB
        I_M["Memory: O(d) Minimal<br/>32 stack frames (< 1 KB!)"]
        I_T["Time: ~52 ms<br/>Depth-first threshold iterations"]
    end
```

### 1. Why IDA* Requires More Work on Road Networks (Graph vs. Tree Complexity)

| Dimension | Forward $A^*$ / Bidirectional $A^*$ | Iterative Deepening $A^*$ (IDA\*) |
| :--- | :--- | :--- |
| **Search Paradigm** | Best-First Search with Priority Queue | Depth-First Search with Dynamic $f$-Cost Cutoffs |
| **Memory Consumption** | $\mathcal{O}(\|V\|)$ ($\approx 10 - 50\text{ KB}$ for local graph) | **$\mathcal{O}(d) \le 32\text{ frames}$ ($< 1\text{ KB}$)** |
| **Node Re-expansion** | **Zero** (nodes closed once with consistent $h$) | **High** (earlier depths re-evaluated each iteration) |
| **Path Redundancy** | Prevented globally by `closed_set` | Explores multiple geometric paths to the same intersection |
| **Optimal Domain** | Road networks, grid navigation, autonomous vehicles | High-branching trees (15-puzzle, Rubik's cube, chess) |

On a 2D road network or planar grid:
1. **Exponential Path Combinations**: Multiple sequences of road segments connect the same two intersections. In depth-first search, without an active global open/closed queue, DFS explores every alternate permutation under the current threshold before pruning.
2. **Iterative Reset Overhead**: Every time the threshold increases ($T_{k+1} \leftarrow \min_{f > T_k} f$), DFS restarts from the start node $s$ from scratch, repeating earlier depth layers.
3. **High Branching Redundancy**: In cyclic road topologies, standard IDA* performs orders of magnitude more node evaluations ($22,346$ evaluations vs $233$ expansions on the Piedmont network).

---

### 2. Real-World Optimizations Applied to Continuous Road Networks

Without specialized continuous-domain adaptations, standard IDA* on floating-point road maps will trigger the *"floating-point epsilon-stepping problem"* ($\epsilon = 10^{-4}\text{ s}$ increments causing millions of iterations). The following optimizations bridge IDA* to spatial networks:

1. **Within-Iteration Dominance Pruning (`best_g`)**:
   - Tracks the lowest $g(v)$ achieved so far during the current threshold iteration to eliminate suboptimal duplicate subtrees without exceeding $\mathcal{O}(V)$ memory.
2. **Active Branch Loop / Cycle Avoidance (`v not in path`)**:
   - Maintains the recursion stack path array, immediately discarding back-edges to ancestors.
3. **Threshold $\delta$-Stepping (`threshold = max(t, threshold + step_size)`)**:
   - Advances continuous travel times in discrete increments ($\delta = 5.0\text{ s}$), preventing micro-iteration thrashing while guaranteeing optimal bounding.
4. **Admissible Haversine Heuristic ($h(u) = d_{\text{Haversine}} / v_{\max}$)**:
   - Accurately guides the depth-first branch directly towards the destination node.

---

### 3. Engineering Recommendations & Trade-Off Summary

- **Bidirectional $A^*$ is the Gold Standard for Autonomous Navigation**:
  - Provides the fastest real-time execution ($0.8\text{ ms}$) and greatest node reduction ($82.9\%$).
- **When IDA* is Justified**:
  - Extremely resource-constrained embedded automotive ECUs (e.g., safety microcontrollers with $< 64\text{ KB}$ RAM) where priority queue allocations $\mathcal{O}(V)$ would cause stack/heap overflow.
- **When $A^*$ with Priority Queue is Preferred**:
  - All standard vehicle compute platforms (NVIDIA DRIVE, Intel Xeon, industrial IPCs) where megabytes of RAM are readily available.


In [ ]:
# Dedicated 3-Way Informed Benchmark on Real-World Piedmont Road Network
t0 = time.perf_counter()
p_a_osm, c_a_osm, exp_a_osm, ord_a_osm = osmnx_astar(G_osm, start_osm, goal_osm, weight_h=1.0, max_speed_mps=max_speed_mps)
t_a_osm = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
p_bi_osm, c_bi_osm, exp_bi_osm, (vis_f_osm, vis_b_osm), meet_osm = osmnx_bidirectional_astar(G_osm, start_osm, goal_osm, max_speed_mps=max_speed_mps)
t_bi_osm = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
p_ida_osm, c_ida_osm, evals_ida_osm, iters_ida_osm = osmnx_ida_star(G_osm, start_osm, goal_osm, max_speed_mps=max_speed_mps, step_size=5.0)
t_ida_osm = (time.perf_counter() - t0) * 1000

# Compile 3-Way Polars Telemetry Table for Piedmont
informed_comp_df = pl.DataFrame({
    "Algorithm": [
        "Forward A* (Standard Informed)", 
        "Bidirectional A* (Dual Fronts)", 
        "Iterative Deepening A* (IDA*)"
    ],
    "Search Paradigm": [
        "Best-First (Priority Queue)", 
        "Dual Simultaneous Best-First", 
        "Depth-First Search (f-Cutoffs)"
    ],
    "Peak Memory State": [
        f"O(V) ~ {exp_a_osm} nodes stored in RAM", 
        f"O(2·b^(d/2)) ~ {exp_bi_osm} nodes stored in RAM", 
        f"O(d) ~ {len(p_ida_osm)} stack frames (< 1 KB!)"
    ],
    "Expansions / Evaluations": [
        exp_a_osm, 
        exp_bi_osm, 
        evals_ida_osm
    ],
    "Iterations": [
        1, 
        1, 
        iters_ida_osm
    ],
    "Optimal Travel Time (s)": [
        f"{c_a_osm:.2f}", 
        f"{c_bi_osm:.2f}", 
        f"{c_ida_osm:.2f}"
    ],
    "Execution Time (ms)": [
        f"{t_a_osm:.2f}", 
        f"{t_bi_osm:.2f}", 
        f"{t_ida_osm:.2f}"
    ]
})

print("=== 3-Way Informed Search Benchmark (Piedmont Road Network = 352 Nodes) ===")
display(informed_comp_df)

assert math.isclose(c_a_osm, c_bi_osm, rel_tol=1e-4)
assert math.isclose(c_a_osm, c_ida_osm, rel_tol=1e-4)
print(f"✅ All 3 informed search algorithms reached identical optimal cost ({c_ida_osm:.2f} s) on Piedmont map!")


In [ ]:
# Plotly 3-Way Informed Visualization on Piedmont Map
edge_x, edge_y = [], []
for u, v, data in G_osm.edges(data=True):
    x0, y0 = G_osm.nodes[u]['x'], G_osm.nodes[u]['y']
    x1, y1 = G_osm.nodes[v]['x'], G_osm.nodes[v]['y']
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

fig_ida_osm = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f"Forward A* ({exp_a_osm} nodes, O(V) RAM)",
        f"Bidirectional A* ({exp_bi_osm} nodes, O(2·b^(d/2)))",
        f"IDA* ({iters_ida_osm} iters, O(d)={len(p_ida_osm)} Stack)"
    ),
    horizontal_spacing=0.05
)

# Background street network on all 3 panels
for col_idx in [1, 2, 3]:
    fig_ida_osm.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(color='#CBD5E1', width=1.0), hoverinfo='none', showlegend=False), row=1, col=col_idx)

# Panel 1: Forward A* Closed Set
fwd_x = [G_osm.nodes[n]['x'] for n in ord_a_osm]
fwd_y = [G_osm.nodes[n]['y'] for n in ord_a_osm]
fig_ida_osm.add_trace(go.Scatter(x=fwd_x, y=fwd_y, mode='markers', marker=dict(color='#38BDF8', size=6, opacity=0.8), name='Forward A* Explored'), row=1, col=1)

# Panel 2: Bidirectional A* Wavefronts
bi_f_x = [G_osm.nodes[n]['x'] for n in vis_f_osm]
bi_f_y = [G_osm.nodes[n]['y'] for n in vis_f_osm]
bi_b_x = [G_osm.nodes[n]['x'] for n in vis_b_osm]
bi_b_y = [G_osm.nodes[n]['y'] for n in vis_b_osm]
fig_ida_osm.add_trace(go.Scatter(x=bi_f_x, y=bi_f_y, mode='markers', marker=dict(color='#38BDF8', size=6, opacity=0.85), name='Forward Wavefront'), row=1, col=2)
fig_ida_osm.add_trace(go.Scatter(x=bi_b_x, y=bi_b_y, mode='markers', marker=dict(color='#F43F5E', size=6, opacity=0.85), name='Backward Wavefront'), row=1, col=2)
if meet_osm is not None:
    fig_ida_osm.add_trace(go.Scatter(x=[G_osm.nodes[meet_osm]['x']], y=[G_osm.nodes[meet_osm]['y']], mode='markers', marker=dict(color='#EAB308', size=12, symbol='star-diamond'), name='Meeting Node'), row=1, col=2)

# Panel 3: IDA* Call-Stack Nodes & Solution Path
ida_stack_x = [G_osm.nodes[n]['x'] for n in p_ida_osm]
ida_stack_y = [G_osm.nodes[n]['y'] for n in p_ida_osm]
fig_ida_osm.add_trace(go.Scatter(x=ida_stack_x, y=ida_stack_y, mode='lines+markers', line=dict(color='#A855F7', width=3, dash='dot'), marker=dict(size=7, color='#9333EA'), name='IDA* Active Stack O(d)'), row=1, col=3)

# Optimal Solution Paths on All Panels
for (col_idx, pth, col) in [(1, p_a_osm, '#0284C7'), (2, p_bi_osm, '#10B981'), (3, p_ida_osm, '#10B981')]:
    px = [G_osm.nodes[n]['x'] for n in pth]
    py = [G_osm.nodes[n]['y'] for n in pth]
    fig_ida_osm.add_trace(go.Scatter(x=px, y=py, mode='lines+markers', line=dict(color=col, width=4), marker=dict(size=4), name='Optimal Solution Path', showlegend=(col_idx==1)), row=1, col=col_idx)

# Start and Goal markers on all panels
for col_idx in [1, 2, 3]:
    fig_ida_osm.add_trace(go.Scatter(x=[G_osm.nodes[start_osm]['x']], y=[G_osm.nodes[start_osm]['y']], mode='markers', marker=dict(size=10, color='#2563EB', symbol='circle'), showlegend=False), row=1, col=col_idx)
    fig_ida_osm.add_trace(go.Scatter(x=[G_osm.nodes[goal_osm]['x']], y=[G_osm.nodes[goal_osm]['y']], mode='markers', marker=dict(size=10, color='#DC2626', symbol='diamond'), showlegend=False), row=1, col=col_idx)

fig_ida_osm.update_layout(
    title_text="<b>Real-World Informed Search Comparison on Piedmont Road Network: Forward A* vs Bidirectional A* vs IDA*</b>",
    template="plotly_white",
    height=480,
    width=1150,
    showlegend=True
)
for col_idx in [1, 2, 3]:
    fig_ida_osm.update_xaxes(showgrid=False, zeroline=False, row=1, col=col_idx)
    fig_ida_osm.update_yaxes(showgrid=False, zeroline=False, scaleanchor=f"x{col_idx if col_idx > 1 else ''}", scaleratio=1.0, row=1, col=col_idx)

fig_ida_osm.show()


### 🎥 Visual Evolution: Forward A* vs. Bidirectional A* vs. IDA* (Piedmont Road Network)

In [ ]:
# Video Generation & Display: Forward A* vs Bidirectional A* vs IDA* Search
generate_video_ida = False  # Set to True to re-render the IDA* comparison video

video_ida_mp4 = "ida_vs_astar_comparison.mp4"
video_ida_gif = "ida_vs_astar_comparison.gif"

if generate_video_ida or not os.path.exists(video_ida_mp4):
    print("🎬 Rendering IDA* vs A* search comparison video...")
    try:
        from generate_ida_star_comparison_video import generate_ida_star_comparison_video
        graph_file = "../data/piedmont_road_network.graphml" if os.path.exists("../data/piedmont_road_network.graphml") else "piedmont_road_network.graphml"
        generate_ida_star_comparison_video(graph_file, video_ida_mp4, video_ida_gif, fps=25)
        print("✅ IDA* comparison video generated successfully!")
    except Exception as e:
        print(f"⚠️ Video generation notice: {e}")

if os.path.exists(video_ida_mp4):
    html_code = f"""
    <div style="text-align: center; margin: 15px auto;">
        <video width="100%" max-width="960px" height="auto" controls autoplay loop muted style="border-radius: 8px; box-shadow: 0 4px 14px rgba(0,0,0,0.25);">
            <source src="{video_ida_mp4}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
        <p style="color: #64748b; font-size: 13px; margin-top: 8px;">
            🎬 <b>Real-World Piedmont Search Wavefronts:</b> Forward A* vs. Bidirectional A* vs. Memory-Bounded IDA* (Cost: 280.6s)
        </p>
    </div>
    """
    display(HTML(html_code))
elif os.path.exists(video_ida_gif):
    display(Image(filename=video_ida_gif, width=960))


---
## Part 5: Broader Search & Motion Planning Paradigms in Self-Driving Vehicles

Beyond classical point-to-point graph search, modern autonomous vehicle software architectures leverage several specialized planning algorithms:

```mermaid
flowchart TD
    subgraph Families["Autonomous Driving Path Planning Algorithm Families"]
        F1["1. Dynamic & Replanning Search"]
        F2["2. Anytime Search"]
        F3["3. Kinodynamic Non-Holonomic Search"]
        F4["4. Continental Scale Preprocessing"]
    end

    F1 --> DStar["<b>D* / D* Lite / LPA*</b><br/>Incremental search: Updates shortest paths when obstacles/traffic changes without recomputing from scratch."]
    F2 --> ARAStar["<b>ARA* (Anytime Repairing A*)</b><br/>Runs high inflation (w=3.0) for an instant feasible path, then progressively refines to optimal."]
    F3 --> HybridAStar["<b>Hybrid A*</b> (Stanford / DARPA)<br/>Searches in continuous vehicle state space (x, y, θ) using Dubins / Reeds-Shepp curves."]
    F4 --> CH["<b>Contraction Hierarchies (CH)</b><br/>Multi-level highway node shortcuts: Sub-millisecond continental queries for cloud navigation."]
```

### Comparative Summary of Motion Planning Algorithms

| Algorithm | Memory Complexity | Re-expansions | Best Suited For | Key Limitation in Driving |
| :--- | :--- | :--- | :--- | :--- |
| **Dijkstra** | $\mathcal{O}(V)$ | None | All-pairs routing / baseline | Explores 100% of network |
| **Forward $A^*$** | $\mathcal{O}(V)$ | None (consistent $h$) | Standard point-to-point routing | High memory on massive maps |
| **Bidirectional $A^*$** | $\mathcal{O}(2 \cdot b^{d/2})$ | None | High-performance route planning | Requires known fixed destination |
| **IDA\*** | **$\mathcal{O}(d)$ (Minimal)** | **High** | Combinatorial puzzles / Microcontrollers | Epsilon-stepping slowdown on float road weights |
| **$D^*$ Lite** | $\mathcal{O}(V)$ | Minimal | Dynamic obstacle replanning | State maintenance overhead |
| **Hybrid $A^*$** | $\mathcal{O}(\text{Cells})$ | Continuous | Autonomous parking & tight maneuvers | Computationally intensive |
